# TrainLM on TPU

This is the same Hugging Face-like workflow an end user runs. Choose a TPU runtime, set the shard count and local cache/output directories below, then call `trainer.train()`. This smoke starts with TrainLM's 135M Llama-shaped reference configuration so model preflight fits comfortably on a v5e-8. TrainLM resolves the dataset revision to an immutable commit before launching workers.

TrainLM handles TPU discovery, world size, worker launch, ranks, preflight, caching, distributed data ownership, checkpoints, evaluation, and structured results under the hood.


## 1. Install and restart once

Kaggle preinstalls TensorFlow and mismatched vision/audio wheels that can initialize or conflict with Torch/XLA in the notebook process. TrainLM text pretraining does not use them. Run the cleanup/install cell, wait for it to finish, then use **Restart Session** before running section 2. Do not continue in the same kernel.


In [ ]:
%pip uninstall -y tensorflow torchvision torchaudio
%pip install -e ".[tpu-xla]" -c constraints/tpu-xla-2.9.txt


## 2. Your inputs

Choose how many numbered shards to use. `(0, TRAIN_SHARD_STOP)` is end-exclusive, so `2` downloads shards `00000` and `00001`. The next shard is reserved for evaluation. TrainLM resolves `main` to its immutable Hub commit and saves the files in `DATA_CACHE_DIR`.

The default model is the from-scratch 135M Llama-shaped reference used by TrainLM's parity work—not the 3.8B Phi checkpoint. After this lifecycle smoke passes, substitute a larger compatible model only with an appropriate sharding plan and HBM budget.


In [ ]:
import importlib.util
from pathlib import Path

assert importlib.util.find_spec("tensorflow") is None, (
    "TensorFlow is still importable. Restart the Kaggle session after section 1."
)

MODEL_CONFIG = {
    "provider": "huggingface",
    "initialization": "config",
    "model_type": "llama",
    "dtype": "float32",
    "config_overrides": {
        "vocab_size": 32064,
        "hidden_size": 1024,
        "intermediate_size": 2816,
        "num_hidden_layers": 8,
        "num_attention_heads": 8,
        "num_key_value_heads": 8,
        "max_position_embeddings": 2048,
        "tie_word_embeddings": True,
        "use_cache": False,
        "_attn_implementation": "sdpa",
    },
}
DATASET_ID = "LaughTaleAI/LaughLM-Tokenized-Fine"
DATASET_REVISION = "main"
TRAIN_SHARD_STOP = 2
DATA_CACHE_DIR = Path("/kaggle/working/huggingface-cache")
OUTPUT_DIR = Path("/kaggle/working/trainlm-run")


## 3. Build the datasets and trainer

`from_hub()` downloads the requested `.bin` range once into `DATA_CACHE_DIR`, scans and validates it before TPU launch, then trains from those local files through lazy memory maps. It does not download during training. Users do not write download loops or configure a dataloader, process count, rank, or world size.


In [ ]:
from trainlm import PackedBinDataset, TrainLMTrainer

sequence_length = 2048
train_dataset = PackedBinDataset.from_hub(
    DATASET_ID,
    revision=DATASET_REVISION,
    shard_range=(0, TRAIN_SHARD_STOP),
    sequence_length=sequence_length,
    cache_dir=DATA_CACHE_DIR,
    split="train",
)
eval_dataset = PackedBinDataset.from_hub(
    DATASET_ID,
    revision=train_dataset.hub_revision,
    shard_range=(TRAIN_SHARD_STOP, TRAIN_SHARD_STOP + 1),
    sequence_length=sequence_length,
    cache_dir=DATA_CACHE_DIR,
    split="validation",
)

trainer = TrainLMTrainer.from_config(
    {
        "api_version": "1",
        "model": MODEL_CONFIG,
        "training_args": {
            "output_dir": OUTPUT_DIR,
            "accelerator": "tpu",
            "bf16": True,
            "max_steps": 6,
            "sequence_length": sequence_length,
            "per_device_train_batch_size": 1,
            "gradient_accumulation_steps": 1,
            "logging_steps": 1,
            "eval_steps": 2,
            "save_steps": 2,
        },
    },
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)


## 4. Train

This single call performs the private collective probe and model preflight, launches every available TPU worker, trains, evaluates every two steps, and writes committed checkpoints every two steps. If the cell is interrupted or a worker fails, TrainLM terminates the complete private worker process group before returning the error, so you can fix the input and rerun. If the failed run already committed a checkpoint, resume it or choose a new output directory instead of overwriting it.

In [ ]:
result = trainer.train()
result

## 5. Optional: inspect what TrainLM selected

`explain()` is useful when reviewing fallbacks or filing a result. It does not require users to inspect worker commands or logs.

In [ ]:
trainer.explain(format="text")

## 6. Optional: resume

Normally set this to the last committed checkpoint after an interrupted or completed run. TrainLM validates topology and restores model, optimizer, scheduler, runtime, RNG, trainer, and packed-data position internally.

In [ ]:
resumed_trainer = TrainLMTrainer.from_config(
    {
        "api_version": "1",
        "model": MODEL_CONFIG,
        "training_args": {
            "output_dir": "/kaggle/working/trainlm-resumed",
            "accelerator": "tpu",
            "bf16": True,
            "max_steps": 10,
            "sequence_length": sequence_length,
            "per_device_train_batch_size": 1,
            "gradient_accumulation_steps": 1,
            "logging_steps": 1,
            "eval_steps": 2,
            "save_steps": 2,
        },
    },
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)
resumed_result = resumed_trainer.train(
    resume_from_checkpoint=OUTPUT_DIR / "checkpoint-4"
)
resumed_result


## What to save from a validation run

Archive the output directory, including `coordinator_summary.json`, `summary.json`, `metrics.jsonl`, committed checkpoint manifests/shards, and XLA metrics. The returned result is the normal user-facing status; these files are only needed for debugging or performance certification.

A successful run validates the lifecycle on that TPU. It does not by itself mark performance as certified.